# Train Reader trên Colab (B1 extractive; sau này R1/R2 ViT5)

Notebook **chỉ gọi `src/`**, không chép code. Chuẩn bị trên máy local:

```
git archive -o hmtt.zip HEAD      # rồi tải hmtt.zip lên Drive: MyDrive/HMTT/hmtt.zip
```
Runtime → GPU (T4 là đủ cho base).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/HMTT'
!rm -rf /content/HMTT && mkdir -p /content/HMTT && cd /content/HMTT && unzip -q $DRIVE/hmtt.zip
%cd /content/HMTT
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Colab đã có torch khớp CUDA -> chỉ cài phần còn lại (transformers 4.57.6 ghim, xem requirements.txt)
!pip install -q -r requirements.txt

**Restart runtime** (Runtime → Restart session) sau khi pip install, rồi chạy tiếp từ ô dưới (không chạy lại ô pip).

In [ ]:
%cd /content/HMTT
import transformers, torch; print(transformers.__version__, torch.__version__, torch.cuda.is_available())
assert transformers.__version__ == '4.57.6', 'Chưa restart runtime?'

## B1 — xlm-roberta-base trích xuất, train trên đoạn đúng, chọn ngưỡng null trên dev

In [ ]:
!PYTHONIOENCODING=utf-8 python src/reader/train_extractive.py --config configs/reader_extractive.yaml --tune_threshold
!PYTHONIOENCODING=utf-8 python src/eval/evaluate_pipeline.py data/predictions/b1_gold_dev.jsonl

Tuỳ chọn — B1 trên đoạn retriever tìm được (cần index: `python src/retriever/build_index.py`, ~4 phút).

In [ ]:
# !PYTHONIOENCODING=utf-8 python src/retriever/build_index.py
# !PYTHONIOENCODING=utf-8 python src/reader/train_extractive.py --predict_only models/b1_xlmr --context retrieved --k 1
# !PYTHONIOENCODING=utf-8 python src/reader/train_extractive.py --predict_only models/b1_xlmr --context retrieved --k 3

In [ ]:
# Lưu kết quả lên Drive (checkpoint KHÔNG đưa vào git)
!mkdir -p $DRIVE/predictions $DRIVE/models
!cp data/predictions/*.jsonl $DRIVE/predictions/
!cp -r models/b1_xlmr $DRIVE/models/